<a href="https://colab.research.google.com/github/Sathvikabanavath/GenAI-Tasks/blob/main/Simplememory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai requests beautifulsoup4

In [ ]:
import requests
from bs4 import BeautifulSoup
from typing import List, Dict
import openai

openai_client = openai.OpenAI(
    api_key="YOUR_API_KEY_HERE"
)

In [ ]:
class Memory:
    def __init__(self):
        self.history: List[Dict[str, str]] = []

    def add(self, role: str, content: str):
        self.history.append({"role": role, "content": content})

    def get(self):
        return self.history


In [ ]:
def fetch_web_content(url: str, max_chars: int = 3000) -> str:
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    soup = BeautifulSoup(response.text, "html.parser")

    for tag in soup(["script", "style"]):
        tag.decompose()

    text = soup.get_text(separator=" ", strip=True)
    return text[:max_chars]

In [ ]:
def summarize(text: str) -> str:
    prompt = f"Summarize the following content in simple bullet points:\n\n{text}"

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )

    return response.choices[0].message.content

In [ ]:
class SimpleAgent:
    def __init__(self):
        self.memory = Memory()

    def run(self, user_input: str):
        self.memory.add("user", user_input)

        if "http" in user_input:
            print("🔧 Tool invoked: fetch_web_content")
            content = fetch_web_content(user_input)

            print("🧠 Tool output received, summarizing...")
            summary = summarize(content)

            self.memory.add("assistant", summary)
            return summary

        else:
            response = "Please provide a URL to fetch and summarize."
            self.memory.add("assistant", response)
            return response

In [ ]:
agent = SimpleAgent()

url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
output = agent.run(url)

print("\n FINAL ANSWER:\n")
print(output)